# DataLead v1 — Interactive API Testing

Cada célula mostra PAYLOAD + RESULTADO

## Cell 0: Setup & Auth

In [1]:
import json, base64, requests, pandas as pd
from io import StringIO

BASE_URL = "https://dev.nettoolpro.cognatis.com.br"
AUTH_URL = f"{BASE_URL}/passport/api/Token/login"
SUMMARY_URL = f"{BASE_URL}/dev/datalead/api/v1/export/summary"

EMAIL = "reinaldo.gregori@cognatis.com.br"
PASSWORD = "Cog2222"
ENV_ID = 1

EXPR_POP = 449
EXPR_RENDA = 6547
CNPJ = 33461874000103
COORDS_SP = {"id": "pt_sp", "x": -46.6333, "y": -23.5505}
ADDR_SP = {"id": "sp_01", "zipCode": 1310100, "number": 100}

def b64(t):
    return base64.b64encode(t.encode()).decode()

def _extrair_mensagem(response):
    try:
        body = response.json()
        if isinstance(body, dict):
            return body.get("message") or body.get("error") or str(body)
    except:
        pass
    return response.text.strip()

print("\n" + "="*70)
print("🔐 AUTENTICANDO...")
print("="*70)

# Etapa 1: login SEM environmentId (obtém token inicial)
print(f"\n1️⃣ POST {AUTH_URL} (sem env)")

payload_1 = {
    "username": b64(EMAIL),
    "password": b64(PASSWORD),
    "loginAttempts": 1
}

r = requests.post(AUTH_URL, headers={"Content-Type": "application/json"}, json=payload_1, timeout=30)

TOKEN = None
if r.status_code != 200:
    print(f"❌ ERRO {r.status_code}: {_extrair_mensagem(r)}")
else:
    data = r.json()
    if data.get("authenticated") is False:
        print(f"❌ Autenticação recusada: {data.get('message', '')}")
    else:
        raw_token = data.get("accessToken") or data.get("token")
        if raw_token:
            TOKEN = raw_token[7:].strip() if raw_token.lower().startswith("bearer ") else raw_token
            print(f"✓ Token inicial obtido")

            # Etapa 2: login COM environmentId (obtém token escopado)
            print(f"\n2️⃣ POST {AUTH_URL} (com env={ENV_ID})")

            payload_2 = {
                "username": b64(EMAIL),
                "password": b64(PASSWORD),
                "loginAttempts": 1,
                "environmentId": ENV_ID
            }

            r2 = requests.post(AUTH_URL, headers={"Content-Type": "application/json"}, json=payload_2, timeout=30)

            if r2.status_code == 200:
                data2 = r2.json()
                raw_token2 = data2.get("accessToken") or data2.get("token")
                if raw_token2:
                    TOKEN = raw_token2[7:].strip() if raw_token2.lower().startswith("bearer ") else raw_token2
                    print(f"✓ Token escopado obtido")
            else:
                print(f"⚠️ Escopo falhou, usando token inicial. {_extrair_mensagem(r2)}")

if TOKEN:
    HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
    print(f"\n✅ AUTENTICADO")
    print(f"  Email: {EMAIL}")
    print(f"  Env ID: {ENV_ID}")
    print(f"  Token: {TOKEN[:30]}...")
    print(f"\n✓ Execute as células de casos de uso abaixo")
    print("="*70)
else:
    print(f"\n❌ Autenticação falhou")


🔐 AUTENTICANDO...

1️⃣ POST https://dev.nettoolpro.cognatis.com.br/passport/api/Token/login (sem env)
✓ Token inicial obtido

2️⃣ POST https://dev.nettoolpro.cognatis.com.br/passport/api/Token/login (com env=1)
✓ Token escopado obtido

✅ AUTENTICADO
  Email: reinaldo.gregori@cognatis.com.br
  Env ID: 1
  Token: eyJhbGciOiJIUzI1NiIsInR5cCI6Ik...

✓ Execute as células de casos de uso abaixo


## Cell 1: Helper Functions

## Cell 1: Helper Functions

In [12]:
def build_payload(dimension=None, select=None, data_input=None, data_where=None, expr_where=None, dry_run=False):
    if select is None:
        select = [{"id": EXPR_POP}, {"id": EXPR_RENDA}]
    if data_input is None:
        data_input = {"keys": [CNPJ] if not isinstance(CNPJ, list) else CNPJ, "geoLevel": 1}
    
    return {
        "compact": False,
        "verbose": False,
        "dryRun": dry_run,
        "delimiter": "|",
        "formatType": "csv",
        "dimension": dimension,
        "expressions": {
            "major": None,
            "moduleId": None,
            "subModule": None,
            "select": select,
            "where": expr_where or [],
            "groupBy": [],
            "having": [],
            "orderBy": [],
            "pageOptions": None
        },
        "data": {
            **data_input,
            "select": [],
            "where": data_where or [],
            "orderBy": [],
            "pageOptions": None
        }
    }

def build_buffer(raios):
    return {"buffer": {"values": [{"distance": d, "alias": f"{d//1000}KM" if d >= 1000 else f"{d}M"} for d in raios]}}

def call_api(payload, label=""):
    print(f"\n{'='*80}")
    if label:
        print(f"📋 {label}")
    print(f"{'='*80}")
    
    print(f"\n📤 PAYLOAD:")
    print(json.dumps(payload, indent=2))
    
    print(f"\n🔄 Chamando API...")
    r = requests.post(SUMMARY_URL, headers=HEADERS, json=payload, timeout=120)
    
    print(f"📥 HTTP {r.status_code}")
    
    if r.status_code != 200:
        print(f"❌ ERRO: {r.text[:500]}")
        return None
    
    try:
        df = pd.read_csv(StringIO(r.text), sep="|")
        print(f"\n✅ {len(df)} linhas × {len(df.columns)} colunas")
        return df,r
    except Exception as e:
        print(f"❌ Erro: {e}")
        return None

print("✓ Helpers prontos")

✓ Helpers prontos


In [ ]:
# Ex 5: Expressões customizadas + filtro em expressions.where
custom_expressions = [
    {"id": EXPR_POP},
    {"id": EXPR_RENDA},
    {"id": 493}  # trabalhadores
]

expr_filter = [{
    "rules": [{"field": "segmento_pj", "operator": "equal", "value": "FARMACIA"}],
    "condition": "and"
}]

payload = build_payload_v2(
    cnpjs=33461874000103,
    expressions=custom_expressions,
    expr_where=expr_filter,
    data_geoLevel=1,
    verbose=True,
    compact=True
)

df = call_api(payload, label="V2 Ex5: Expressões custom + expr_where filter + compact+verbose")
if df is not None:
    print(f"✅ {len(df)} linhas")
    display(df)

In [ ]:
# Ex 4: CSV file upload automático
# Primeiro cria um CSV temp pra teste

csv_content = """id,zipCode,number
test_sp,1310100,100
test_rj,22071900,50
test_bh,30112000,200"""

with open('/tmp/test_addresses.csv', 'w') as f:
    f.write(csv_content)

# Agora usa na função
payload = build_payload_v2(
    csv_file='/tmp/test_addresses.csv',
    upload=True,
    data_geoLevel=49
)

df = call_api(payload, label="V2 Ex4: CSV upload automático + geoLevel=49")
if df is not None:
    print(f"✅ {len(df)} linhas")
    display(df)

In [ ]:
# Ex 3: Filtro PJ via data.where + dimension.geolevels
bradesco_filter = [{
    "rules": [{"field": "razaosocial", "operator": "contains", "value": "BRADESCO"}],
    "condition": "and"
}]

payload = build_payload_v2(
    data_where=bradesco_filter,
    dimension_geolevels=[49, 3, 5],
    verbose=False
)

df = call_api(payload, label="V2 Ex3: Filtro PJ (Bradesco) + dimension.geolevels (49, 3, 5)")
if df is not None:
    print(f"✅ {len(df)} linhas")
    if 'area' in df.columns:
        print(f"   Níveis: {df['area'].unique()}")
    display(df.head())

In [ ]:
# Ex 2: Múltiplos endereços + buffer + verbose
addrs = [
    {"id": "sp_01", "zipCode": 1310100, "number": 100},
    {"id": "rj_01", "zipCode": 22071900, "number": 50},
    {"id": "bh_01", "zipCode": 30112000, "number": 200}
]

payload = build_payload_v2(
    addresses=addrs,
    dimension_buffer_distances=[500, 1000],
    verbose=True
)

df = call_api(payload, label="V2 Ex2: 3 endereços + buffer 500m/1km + verbose")
if df is not None:
    print(f"✅ {len(df)} linhas")
    display(df.head())

In [ ]:
# Ex 1: Múltiplos CNPJs + data.geoLevel
payload = build_payload_v2(
    cnpjs=[33461874000103, 5951509000133, 123456789],
    data_geoLevel=49,
    verbose=False
)

df = call_api(payload, label="V2 Ex1: 3 CNPJs + geoLevel=49")
if df is not None:
    print(f"✅ {len(df)} linhas")
    display(df)

---

# build_payload_v2 — Exemplos de Uso

In [ ]:
def build_payload_v2(
    # Input data
    cnpjs=None,
    addresses=None,
    coordinates=None,
    csv_file=None,
    upload=False,
    
    # Expressions (select)
    expressions=None,
    
    # Filtering
    data_where=None,
    expr_where=None,
    
    # Dimensions
    dimension_geolevels=None,
    dimension_buffer_distances=None,
    
    # Data aggregation — OPTIONAL, only add if specified
    data_geoLevel=None,
    
    # API options
    verbose=False,
    dry_run=False,
    compact=False
):
    """
    Flexible payload builder supporting all DataLead v1 patterns.
    
    Args:
        cnpjs: int or list of ints (CNPJs)
        addresses: dict or list of dicts with {id, zipCode, number}
        coordinates: dict or list of dicts with {id, x, y}
        csv_file: path to local CSV file to upload
        upload: bool, if True and csv_file provided, uploads CSV and embeds fileRef
        expressions: list of {"id": ...} or defaults to [pop, renda]
        data_where: list of filter rules for data input
        expr_where: list of filter rules for expression aggregation
        dimension_geolevels: list of geoLevel IDs [49, 3, 5] for grouping sets
        dimension_buffer_distances: list of distances in meters [500, 1000, 1500]
        data_geoLevel: int, aggregation level (OPTIONAL — only add if specified!)
        verbose: bool
        dry_run: bool
        compact: bool
    
    Returns:
        dict: payload ready for API POST
    """
    
    # Default expressions
    if expressions is None:
        expressions = [{"id": EXPR_POP}, {"id": EXPR_RENDA}]
    
    # Build data input
    data_input = {}
    
    if cnpjs is not None:
        if isinstance(cnpjs, int):
            cnpjs = [cnpjs]
        data_input["keys"] = cnpjs
    
    elif addresses is not None:
        if isinstance(addresses, dict):
            addresses = [addresses]
        data_input["addresses"] = addresses
    
    elif coordinates is not None:
        if isinstance(coordinates, dict):
            coordinates = [coordinates]
        data_input["values"] = coordinates
    
    elif csv_file is not None:
        if upload:
            with open(csv_file, 'r') as f:
                csv_content = f.read()
            fileRef = upload_csv(csv_content, filename=csv_file.split('/')[-1])
            if fileRef:
                data_input["file"] = {
                    "url": fileRef,
                    "provider": "GoogleCloudBucket",
                    "contentType": "Addresses",
                    "delimiter": ",",
                    "fields": {"id": "id", "zipCode": "zipCode", "number": "number"}
                }
        else:
            print("⚠️  csv_file provided but upload=False. Skipping file upload.")
    
    # Add geoLevel ONLY if explicitly specified (data_geoLevel is optional!)
    if data_geoLevel is not None:
        data_input["geoLevel"] = data_geoLevel
    
    # Build dimension
    dimension = None
    
    if dimension_geolevels is not None:
        if isinstance(dimension_geolevels, int):
            dimension_geolevels = [dimension_geolevels]
        dimension = {
            "geoLevel": {
                "values": [{"id": gl} for gl in dimension_geolevels]
            }
        }
    
    elif dimension_buffer_distances is not None:
        if isinstance(dimension_buffer_distances, int):
            dimension_buffer_distances = [dimension_buffer_distances]
        dimension = build_buffer(dimension_buffer_distances)
    
    # Build final payload
    payload = {
        "compact": compact,
        "verbose": verbose,
        "dryRun": dry_run,
        "delimiter": "|",
        "formatType": "csv",
        "dimension": dimension,
        "expressions": {
            "major": None,
            "moduleId": None,
            "subModule": None,
            "select": expressions,
            "where": expr_where or [],
            "groupBy": [],
            "having": [],
            "orderBy": [],
            "pageOptions": None
        },
        "data": {
            **data_input,
            "select": [],
            "where": data_where or [],
            "orderBy": [],
            "pageOptions": None
        }
    }
    
    return payload

print("✓ build_payload_v2 atualizado")

## 📌 Exemplo 1: CNPJ geolevel=1

In [13]:
CNPJ = 33461874000103
payload = build_payload(
    dimension=None,
    select=[{"id": EXPR_POP}, {"id": EXPR_RENDA}],
    data_input={"keys": [ CNPJ], "geoLevel": 5}
)

df,r = call_api(payload, label="Exemplo 1: CNPJ geolevel=1 (sem buffer)")

if df is not None:
    print(f"\n📊 RESULTADO:")
    display(df)


📋 Exemplo 1: CNPJ geolevel=1 (sem buffer)

📤 PAYLOAD:
{
  "compact": false,
  "verbose": false,
  "dryRun": false,
  "delimiter": "|",
  "formatType": "csv",
  "dimension": null,
  "expressions": {
    "major": null,
    "moduleId": null,
    "subModule": null,
    "select": [
      {
        "id": 449
      },
      {
        "id": 6547
      }
    ],
    "where": [],
    "groupBy": [],
    "having": [],
    "orderBy": [],
    "pageOptions": null
  },
  "data": {
    "keys": [
      33461874000103
    ],
    "geoLevel": 5,
    "select": [],
    "where": [],
    "orderBy": [],
    "pageOptions": null
  }
}

🔄 Chamando API...
📥 HTTP 200

✅ 1 linhas × 1 colunas

📊 RESULTADO:


,"ï»¿""pop_total_abs"";""renda_media_fam_SM"""
0,";"""""


## 📌 Exemplo 2: CNPJ + Buffer (3 raios)

In [14]:
EXPR_POP = 449
EXPR_RENDA = 6547
EXPR_TRAB = 493
payload = build_payload(
    dimension=build_buffer([500, 1000, 1500]),
    select=[{"id": EXPR_POP}, {"id": EXPR_RENDA}, {"id": EXPR_TRAB}],
    data_input={"keys": [33461874000103, 5951509000133]}  # Pass as list directly
)

df,r = call_api(payload, label="Exemplo 2: CNPJ + Buffer 500m/1km/1.5km (2 CNPJs)")

if df is not None:
    print(f"\n📊 RESULTADO ({len(df)} linhas):")
    display(df)


📋 Exemplo 2: CNPJ + Buffer 500m/1km/1.5km (2 CNPJs)

📤 PAYLOAD:
{
  "compact": false,
  "verbose": false,
  "dryRun": false,
  "delimiter": "|",
  "formatType": "csv",
  "dimension": {
    "buffer": {
      "values": [
        {
          "distance": 500,
          "alias": "500M"
        },
        {
          "distance": 1000,
          "alias": "1KM"
        },
        {
          "distance": 1500,
          "alias": "1KM"
        }
      ]
    }
  },
  "expressions": {
    "major": null,
    "moduleId": null,
    "subModule": null,
    "select": [
      {
        "id": 449
      },
      {
        "id": 6547
      },
      {
        "id": 493
      }
    ],
    "where": [],
    "groupBy": [],
    "having": [],
    "orderBy": [],
    "pageOptions": null
  },
  "data": {
    "keys": [
      33461874000103,
      5951509000133
    ],
    "select": [],
    "where": [],
    "orderBy": [],
    "pageOptions": null
  }
}

🔄 Chamando API...
📥 HTTP 200

✅ 6 linhas × 1 colunas

📊 RESULTADO (

,"ï»¿""id"";""area"";""pop_total_abs"";""renda_media_fam_SM"";""qtd_trabalhadores_abs"""
0,"5951509000133;""500M"";""14133"";""27.55"";""55207"""
1,"5951509000133;""1KM"";""38075"";""29.47"";""106636"""
2,"5951509000133;""1KM"";""62121"";""30.01"";""216618"""
3,"33461874000103;""500M"";""19476"";""2.95"";""660"""
4,"33461874000103;""1KM"";""53981"";""3.01"";""3010"""
5,"33461874000103;""1KM"";""108991"";""3.17"";""9471"""


In [29]:
r.text

'ï»¿"id";"area";"pop_total_abs";"renda_media_fam_SM";"qtd_trabalhadores_abs"\r\n"5951509000133";"500M";"14133";"27.55";"55207"\r\n"5951509000133";"1KM";"38075";"29.47";"106636"\r\n"5951509000133";"1KM";"62121";"30.01";"216618"\r\n"33461874000103";"500M";"19476";"2.95";"660"\r\n"33461874000103";"1KM";"53981";"3.01";"3010"\r\n"33461874000103";"1KM";"108991";"3.17";"9471"\r\n'

## 📌 Exemplo 3: Coordenadas + Buffer

In [16]:
EXPR_POP = 449
EXPR_RENDA = 6547
EXPR_TRAB = 493
CNPJ = 33461874000103
COORDS_SP = [{"id": "pt_sp", "x": -46.6333, "y": -23.5505},{"id": "pt_sp2", "x": -49.6333, "y": -20.5505}]


In [17]:
payload = build_payload(
    dimension=build_buffer([3000, 5000]),
    select=[{"id": EXPR_POP}, {"id": EXPR_RENDA}, {"id": EXPR_TRAB}],
    data_input={"values": COORDS_SP}
)

df,r = call_api(payload, label="Exemplo 3: Coordenadas + Buffer 3km/5km")

if df is not None:
    print(f"\n📊 RESULTADO ({len(df)} linhas):")
    display(df)


📋 Exemplo 3: Coordenadas + Buffer 3km/5km

📤 PAYLOAD:
{
  "compact": false,
  "verbose": false,
  "dryRun": false,
  "delimiter": "|",
  "formatType": "csv",
  "dimension": {
    "buffer": {
      "values": [
        {
          "distance": 3000,
          "alias": "3KM"
        },
        {
          "distance": 5000,
          "alias": "5KM"
        }
      ]
    }
  },
  "expressions": {
    "major": null,
    "moduleId": null,
    "subModule": null,
    "select": [
      {
        "id": 449
      },
      {
        "id": 6547
      },
      {
        "id": 493
      }
    ],
    "where": [],
    "groupBy": [],
    "having": [],
    "orderBy": [],
    "pageOptions": null
  },
  "data": {
    "values": [
      {
        "id": "pt_sp",
        "x": -46.6333,
        "y": -23.5505
      },
      {
        "id": "pt_sp2",
        "x": -49.6333,
        "y": -20.5505
      }
    ],
    "select": [],
    "where": [],
    "orderBy": [],
    "pageOptions": null
  }
}

🔄 Chamando API...
📥 H

,"ï»¿""id"";""area"";""pop_total_abs"";""renda_media_fam_SM"";""qtd_trabalhadores_abs"""
0,"pt_sp;""3KM"";""482693"";""11.09"";""690033"""
1,"pt_sp;""5KM"";""955137"";""13.77"";""1183689"""
2,"pt_sp2;""3KM"";""464"";""3.41"";""691"""
3,"pt_sp2;""5KM"";""723"";""3.32"";""691"""


## 📌 Exemplo 4: Endereços + Buffer

In [18]:
ADDR_SP = [ {"recId": "sp01", "zipCode": 1310100, "number": 100},{"recId": "spHOME", "zipCode": 1428000, "number": 276}] 
payload = build_payload(
    dimension=build_buffer([500, 1000]),
    select=[{"id": EXPR_POP}, {"id": EXPR_RENDA}],
    data_input={"addresses": ADDR_SP,"geolevel":49}
)

df,r = call_api(payload, label="Exemplo 4: Endereços + Buffer 500m/1km")

if df is not None:
    print(f"\n📊 RESULTADO ({len(df)} linhas):")
    display(df)


📋 Exemplo 4: Endereços + Buffer 500m/1km

📤 PAYLOAD:
{
  "compact": false,
  "verbose": false,
  "dryRun": false,
  "delimiter": "|",
  "formatType": "csv",
  "dimension": {
    "buffer": {
      "values": [
        {
          "distance": 500,
          "alias": "500M"
        },
        {
          "distance": 1000,
          "alias": "1KM"
        }
      ]
    }
  },
  "expressions": {
    "major": null,
    "moduleId": null,
    "subModule": null,
    "select": [
      {
        "id": 449
      },
      {
        "id": 6547
      }
    ],
    "where": [],
    "groupBy": [],
    "having": [],
    "orderBy": [],
    "pageOptions": null
  },
  "data": {
    "addresses": [
      {
        "recId": "sp01",
        "zipCode": 1310100,
        "number": 100
      },
      {
        "recId": "spHOME",
        "zipCode": 1428000,
        "number": 276
      }
    ],
    "geolevel": 49,
    "select": [],
    "where": [],
    "orderBy": [],
    "pageOptions": null
  }
}

🔄 Chamando API...
📥

,"ï»¿""id"";""area"";""pop_total_abs"";""renda_media_fam_SM"""
0,"sp01;""500M"";""16092"";""21.66"""
1,"sp01;""1KM"";""74877"";""21.19"""
2,"spHOME;""500M"";""9313"";""29.01"""
3,"spHOME;""1KM"";""32559"";""29.67"""


In [ ]:
## 📌 Exemplo 6B: Endereços + geoLevel=49 (Setor Censitário)

In [ ]:
payload = build_payload(
    dimension=None,  # No dimension when using geoLevel
    select=[{"id": EXPR_POP}, {"id": EXPR_RENDA}],
    data_input={
        "addresses": [ADDR_SP],
        "geoLevel": 49  # ← Directly in data, not in dimension
    }
)

print("\n💡 geoLevel=49 (Setor Censitário) vai em 'data', não em 'dimension'")

df = call_api(payload, label="Exemplo 6B: Endereços + geoLevel=49")

if df is not None:
    display(df)

In [51]:


ADDR_SP = [ {"recId": "sp01", "zipCode": 1310100, "number": 100},{"recId": "spHOME", "zipCode": 1428000, "number": 276}] 
payload = build_payload(
    select=[{"id": EXPR_POP}, {"id": EXPR_RENDA}],
    data_input={"addresses": ADDR_SP,"geoLevel":49}
)

df,r = call_api(payload, label="Exemplo 4: Endereços + Buffer 500m/1km")

if df is not None:
    print(f"\n📊 RESULTADO ({len(df)} linhas):")
    display(df)


📋 Exemplo 4: Endereços + Buffer 500m/1km

📤 PAYLOAD:
{
  "compact": false,
  "verbose": false,
  "dryRun": false,
  "delimiter": "|",
  "formatType": "csv",
  "dimension": null,
  "expressions": {
    "major": null,
    "moduleId": null,
    "subModule": null,
    "select": [
      {
        "id": 449
      },
      {
        "id": 6547
      }
    ],
    "where": [],
    "groupBy": [],
    "having": [],
    "orderBy": [],
    "pageOptions": null
  },
  "data": {
    "addresses": [
      {
        "recId": "sp01",
        "zipCode": 1310100,
        "number": 100
      },
      {
        "recId": "spHOME",
        "zipCode": 1428000,
        "number": 276
      }
    ],
    "geoLevel": 49,
    "select": [],
    "where": [],
    "orderBy": [],
    "pageOptions": null
  }
}

🔄 Chamando API...
📥 HTTP 500
❌ ERRO: System.NullReferenceException: Object reference not set to an instance of an object.
   at Cognatis.Database.Infrastructure.Services.BuilderService.CreateMetadataObjects(List`1 m

TypeError: cannot unpack non-iterable NoneType object

## 📌 Exemplo 5: DRY RUN (estimativa)

In [22]:
df

,"ï»¿""id"";""area"";""pop_total_abs"";""renda_media_fam_SM"""
0,"500M;""500M"";""16092"";""21.66"""
1,"1KM;""1KM"";""74877"";""21.19"""


In [ ]:
payload = build_payload(
    dimension=build_buffer([3000]),
    select=[{"id": EXPR_POP}, {"id": EXPR_RENDA}],
    data_input={"addresses": [ADDR_SP]},
    dry_run=True
)

df = call_api(payload, label="Exemplo 5: DRY RUN (estimativa, sem extrair)")

if df is not None:
    display(df)

## 📌 Exemplo 6: data.where (filtro INPUT)

In [52]:
payload = build_payload(
    dimension=None,
    select=[{"id": EXPR_POP}, {"id": EXPR_RENDA}],
    data_input={"keys": [CNPJ], "geoLevel": 1},
    data_where=[{
        "rules": [{"field": "segmento", "operator": "equal", "value": "FARMACIA"}],
        "condition": "and"
    }]
)

print("\n💡 data.where filtra a tabela INPUT (antes de agregar)")

df = call_api(payload, label="Exemplo 6: data.where (filtro INPUT)")

if df is not None:
    display(df)


💡 data.where filtra a tabela INPUT (antes de agregar)

📋 Exemplo 6: data.where (filtro INPUT)

📤 PAYLOAD:
{
  "compact": false,
  "verbose": false,
  "dryRun": false,
  "delimiter": "|",
  "formatType": "csv",
  "dimension": null,
  "expressions": {
    "major": null,
    "moduleId": null,
    "subModule": null,
    "select": [
      {
        "id": 449
      },
      {
        "id": 6547
      }
    ],
    "where": [],
    "groupBy": [],
    "having": [],
    "orderBy": [],
    "pageOptions": null
  },
  "data": {
    "keys": [
      33461874000103
    ],
    "geoLevel": 1,
    "select": [],
    "where": [
      {
        "rules": [
          {
            "field": "segmento",
            "operator": "equal",
            "value": "FARMACIA"
          }
        ],
        "condition": "and"
      }
    ],
    "orderBy": [],
    "pageOptions": null
  }
}

🔄 Chamando API...
📥 HTTP 400
❌ ERRO: ["Existem filtros de referência sem Id de expressão definido (WHERE)","Existem filtros de refer

## 📌 Exemplo 7: expressions.where (filtro AGREGAÇÃO)

In [ ]:
payload = build_payload(
    dimension=build_buffer([3000]),
    select=[{"id": EXPR_POP}, {"id": EXPR_RENDA}],
    data_input={"values": [COORDS_SP]},
    expr_where=[{
        "rules": [{"field": "segmento_pj", "operator": "equal", "value": "FARMACIA"}],
        "condition": "and"
    }]
)

print("\n💡 expressions.where filtra durante a AGREGAÇÃO")

df = call_api(payload, label="Exemplo 7: expressions.where (filtro AGREGAÇÃO)")

if df is not None:
    display(df)

## 📌 Exemplo 8: PS + EX (agregação explícita)

In [ ]:
payload = build_payload(
    dimension=build_buffer([3000]),
    select=[
        {"id": EXPR_POP},
        {"id": EXPR_RENDA},
        {"id": 999, "function": "avg"}  # ⚠️  PS fictício
    ],
    data_input={"values": [COORDS_SP]}
)

print("\n💡 PS (Primitiva Simples) precisa de 'function' quando mistura com EX")
print("   ID 999 é fictício! Substituir por ID real de PS\n")

df = call_api(payload, label="Exemplo 8: PS + EX (com agregação)")

if df is not None:
    display(df)
else:
    print("\n⚠️  Esperado falhar (ID 999 é fictício)")

In [ ]:
# Caso 7: Filtro PJ (Bradesco) + dimension.geoLevel com múltiplos níveis
# Usa grouping sets para retornar todas as combinações

payload = {
    "compact": False, "verbose": False, "dryRun": False,
    "delimiter": "|", "formatType": "csv",
    "dimension": {
        "geoLevel": {
            "values": [
                {"id": 49},   # Setor Censitário
                {"id": 3},    # Bairro
                {"id": 5}     # Município
            ]
        }
    },
    "expressions": {
        "major": None, "moduleId": None, "subModule": None,
        "select": [{"id": EXPR_POP}, {"id": EXPR_RENDA}],
        "where": [], "groupBy": [], "having": [], "orderBy": [], "pageOptions": None
    },
    "data": {
        "select": [],
        "where": [{
            "rules": [{"field": "razaosocial", "operator": "contains", "value": "BRADESCO"}],
            "condition": "and"
        }],
        "orderBy": [], "pageOptions": None
    }
}

print("\n💡 Caso 7: dimension.geoLevel com múltiplos IDs usa grouping sets")
print("   Retorna todas as combinações de níveis (setor, bairro, município)\n")

df = call_api(payload, label="Caso 7: Filtro PJ (Bradesco) + dimension.geoLevel (49, 3, 5)")

if df is not None:
    print(f"\n📊 RESULTADO ({len(df)} linhas):")
    print(f"   Colunas: {list(df.columns)}")
    if 'area' in df.columns:
        print(f"   Níveis únicos em 'area': {df['area'].unique()}")
    display(df.head(10))

---

# 7 NOVOS CASOS — dimension.geoLevel + Filtro PJ

## Caso 7: Filtro PJ (Bradesco) + dimension.geoLevel (múltiplos níveis)